# Lab 1: Document Indexing for RAG Systems

### 1. Overview
In this lab, we build the foundation of a Retrieval-Augmented Generation (RAG) system: the **Indexing Pipeline**.

**The Architecture:**
1.  **PDF Loading:** Extracting raw text from the 100-page manual using `pypdf`.
2.  **Text Splitting (Chunking):** Breaking the text into smaller, overlapping segments to maintain context and stay within model token limits.
3.  **Embeddings:** Converting text chunks into high-dimensional vectors that represent semantic meaning.
4.  **Vector Storage:** Saving these vectors in **ChromaDB** for efficient similarity searching.

In [5]:
# Install necessary libraries with specific text-splitter package
!pip install -q pypdf langchain langchain-community chromadb sentence-transformers langchain-text-splitters

In [6]:
import os
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
# Updated import path for compatibility
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# Step 1: Upload the PDF
print("Please upload your 100-page PDF manual:")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_filename}")

Please upload your 100-page PDF manual:


Saving 2019BurkovTheHundred-pageMachineLearning.pdf to 2019BurkovTheHundred-pageMachineLearning.pdf
Uploaded: 2019BurkovTheHundred-pageMachineLearning.pdf


In [7]:
# Step 2: Load and Split the PDF
# Note: If this still hangs, the PDF might be partially corrupted or too complex.
try:
    loader = PyPDFLoader(pdf_filename)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        length_function=len,
        is_separator_regex=False,
    )

    chunks = text_splitter.split_documents(documents)
    print(f"Successfully split {len(documents)} pages into {len(chunks)} chunks.")
except Exception as e:
    print(f"An error occurred during loading: {e}")

Successfully split 152 pages into 379 chunks.


In [8]:
# Step 3: Initialize Embeddings
# Using a reliable open-source model from HuggingFace (all-MiniLM-L6-v2)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

/tmp/ipykernel_10645/1467856911.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
# Step 4: Create and Persist the Vector Database
# We will store the index in a local directory named 'db_index'

persist_directory = 'db_index'

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector database created and saved to {persist_directory}")

Vector database created and saved to db_index


In [10]:
# Step 5: Verification Test
# Test the index with a sample similarity search

query = "What are the safety instructions?" # Replace with a query relevant to your PDF
docs = vectordb.similarity_search(query, k=3)

print(f"\nTop {len(docs)} relevant chunks found:\n")
for i, doc in enumerate(docs):
    print(f"--- Result {i+1} ---")
    print(doc.page_content[:300] + "...")
    print(f"Source: Page {doc.metadata['page']}\n")


Top 3 relevant chunks found:

--- Result 1 ---
including ones you programmed yourself.
5.7.1 Cross-V alidation
When you don’t have a decent validation set to tune your hyperparameters on, the common
technique that can help you is called cross-validation. When you have few training examples,
it could be prohibitive to have both validation and tes...
Source: Page 70

--- Result 2 ---
The documentation also provides information on hyperparameters.
3If it’s really necessary, the score for SVM and kNN predictions could be synthetically created using some
simple techniques.
Andriy Burkov The Hundred-Page Machine Learning Book - Draft 9...
Source: Page 50

--- Result 3 ---
in a speciﬁc topic covered in the book and want to know more, most sections have a QR
code. By scanning one of those QR codes with your phone, you will get a link to a page on
the book’s companion wiki theMLbook.com with additional materials: recommended reads,
videos, Q&As, code snippets, tutorials...
Source: Page 3



### Troubleshooting Tips

1.  **Out of Memory (OOM):** If Colab crashes, it's likely due to the large PDF. Ensure you are using `RecursiveCharacterTextSplitter` which is memory efficient. If needed, process pages in batches.
2.  **`pypdf` errors:** Some PDFs have complex encodings. If `PyPDFLoader` fails, try installing `pdfminer.six` and using `PDFMinerLoader` instead.
3.  **SQLite3 Version:** ChromaDB requires a recent version of SQLite. If you see a version error, restart your Colab runtime; Colab's default environment is usually compatible now.
4.  **Missing Metadata:** Ensure you check `doc.metadata` if you need to reference specific page numbers during retrieval.